5. Protéger les personnes et organiser le travail
Dans l'équipe, qui a besoin de voir quelles données et comment limiter les accès ? Un patient signale une erreur dans son dossier. Quel droit peut-il exercer et vers qui l'orienter ?

Commit attendu — docs: ajouter chapitre 5 - protection

# Chapitre 5 : Protéger les personnes et organiser le travail

La conformité au RGPD ne repose pas uniquement sur des textes juridiques : elle exige une **organisation rigoureuse** des accès techniques (la sécurité des données) et une **procédure claire** pour garantir aux patients le contrôle sur leurs propres informations.

---

## 1. Cloisonnement des accès : Qui voit quoi dans l'équipe ?

Selon le principe de **sécurité et de confidentialité (Article 32 du RGPD)** et le principe du **moindre privilège**, chaque membre de l'équipe projet ne doit accéder qu'aux données strictement indispensables à sa mission. 

Pour concilier l'entraînement de l'IA et le suivi médical réel, l'hôpital doit mettre en place une **séparation hermétique entre l'identité du patient et ses données médicales** (pseudonymisation).

### Matrice des accès par rôle

| Rôle dans l'équipe | Données accessibles | Données bloquées | Justification |
| :--- | :--- | :--- | :--- |
| **Administrateur Base de Données (DBA) / Service IT** | Données techniques et clés de chiffrement (journaux système). | Contenu médical détaillé et résultats des prédictions. | Maintient l'infrastructure sans avoir à lire les dossiers médicaux. |
| **Data Scientists / Développeurs IA** | **Données pseudonymisées uniquement** (ex: `PAT_8942`, tranche d'âge, codes CIM-10, durée de séjour). | **Identité directe** (Nom, prénom, téléphone, adresse, N° de registre national) et table de correspondance. | L'algorithme apprend sur des profils statistiques ; il n'a pas besoin de connaître le nom du patient. |
| **Médecin Coordinateur / Infirmière de suivi** | **Identité complète + Score de risque IA + Dossier médical** (uniquement pour les patients de son service). | Accès aux patients des autres services hospitaliers non concernés. | Soumis au **secret médical** (Art. 9.3), ce personnel a besoin du nom et du téléphone pour appeler le patient et valider la décision médicale. |
| **DPO (Délégué à la Protection des Données)** | Registre des traitements, journaux d'audit (logs), documentation de sécurité. | Dossiers médicaux nominatifs des patients. | Contrôle la conformité et la traçabilité des accès sans violer le secret médical. |

### Comment limiter techniquement ces accès ?
* **Contrôle d'accès basé sur les rôles (RBAC) :** Création de groupes de droits distincts dans la base de données (ex: rôle `ROLE_IA_READONLY` qui pointe uniquement vers une vue SQL pseudonymisée).
* **Table de correspondance isolée :** Le fichier reliant `PAT_8942` au vrai patient est stocké sur un serveur sécurisé distinct, accessible uniquement au personnel soignant habilité.
* **Traçabilité (Logging) :** Chaque consultation d'un dossier ou export de données est enregistré (qui a vu quoi et quand) afin de détecter toute anomalie ou accès abusif.

---

## 2. Gestion des erreurs : Quel droit pour le patient et vers qui l'orienter ?

Un patient signale une erreur dans son dossier médical (par exemple : une mauvaise date de sortie, un diagnostic erroné ou un antécédent qui ne le concerne pas). Cette situation est critique car **une donnée fausse fausse également la prédiction de l'IA** (principe d'exactitude des données, **Article 5.1.d**).

### Quel droit peut-il exercer ?
Le patient peut exercer son **Droit de rectification (Article 16 du RGPD)** :
* Il a le droit d'obtenir de l'hôpital, dans les meilleurs délais (maximum **1 mois** selon l'Article 12.3), que les données inexactes le concernant soient corrigées.
* Il a également le droit d'obtenir que les données incomplètes soient complétées.
* **Conséquence pour le projet :** Une fois le dossier source corrigé, la mise à jour doit être répercutée dans la base utilisée par l'IA (et le patient doit pouvoir demander la réévaluation de son score de risque).

### Vers qui l'orienter ?
L'équipe projet (développeurs, data scientists) **ne doit jamais modifier manuellement** un dossier médical à la demande d'un patient. Il faut suivre le circuit officiel de l'hôpital :

1. **Pour la demande administrative / RGPD → Le DPO (Délégué à la Protection des Données) ou le Service de Médiation / Droits des patients :**
   C'est le point de contact officiel chargé d'enregistrer la demande formelle de rectification, de vérifier l'identité du demandeur et de garantir la réponse dans le délai légal d'un mois.
2. **Pour la validation médicale de la correction → Le Médecin traitant ou le Médecin-Chef du service concerné :**
   En milieu hospitalier, un diagnostic médical ne peut être modifié que par un médecin (qui peut ajouter une note rectificative tout en conservant la traçabilité de l'historique médical pour des raisons légales).

---

## 3. Cas pratiques : Organisation des accès et droits des patients

### ✅ Exemple 1 : Quand est-ce AUTORISÉ ? (Le scénario conforme)
* **Organisation :** Les data scientists travaillent sur un environnement sécurisé alimenté par une vue SQL ne contenant que des identifiants hachés (`SHA-256` avec sel). Lorsqu'un patient présente un risque élevé (`85 %`), le système transmet l'identifiant haché au logiciel médical de l'infirmière de suivi, qui est la seule habilitée à le décoder pour passer l'appel.
* **Gestion d'une erreur :** Un patient appelle le secrétariat en signalant qu'on lui a attribué par erreur un diabète de type 2. La secrétaire l'oriente vers le **DPO de l'hôpital** (via le formulaire dédié sur le site de l'hôpital) et alerte le **médecin du service**. Le médecin corrige le dossier, et la synchronisation nocturne met automatiquement à jour le profil statistique dans l'outil IA.

> **Verdict : AUTORISÉ.** Le cloisonnement protège l'identité du patient, le droit de rectification est respecté et l'IA travaille sur des données exactes.

---

### ❌ Exemple 2 : Quand est-ce INTERDIT ? (Le scénario illégal)
* **Organisation :** Toute l'équipe projet (y compris les stagiaires développeurs et le chef de projet externe) utilise un compte administrateur partagé (`admin_projet`) pour se connecter directement à la base de production de l'hôpital, avec accès en clair aux noms, numéros de téléphone et diagnostics de tous les patients.
* **Gestion d'une erreur :** Un patient envoie un e-mail à un développeur de l'équipe projet pour signaler une erreur dans sa date de naissance. Le développeur modifie directement la ligne dans la table SQL à la main, sans trace ni validation médicale, et oublie de le signaler au service des admissions.

> **Verdict : STRICTEMENT INTERDIT.**
> * **Violation de l'Art. 32 (Sécurité) :** Compte partagé (absence de traçabilité) et accès non cloisonné aux données nominatives pour du personnel non soignant.
> * **Violation des Art. 12 et 16 (Droits des personnes) :** Absence de procédure officielle via le DPO, risque d'altération de l'intégrité du dossier médical et absence de traçabilité légale.
